# Understanding Forward Pass, Loss, Chain Rule & Backpropagation

In this session, we will open the "black box" of neural networks and trace the exact math step by step.

### Roadmap
1. **Part 1: 3-Layer Network (1 Neuron per Layer)** — Fixed integer weights, step-by-step forward pass, final output, and the full nested equation.
2. **Part 2: 3-Layer Network (2–3 Neurons per Layer)** — Scaling up to matrices while keeping the exact same forward pass idea.
3. **Part 3: The Loss Function** — Measuring how wrong our network's prediction is.
4. **Part 4: The Chain Rule of Derivatives** — A simple, intuitive building block (like connected gears).
5. **Part 5: Backpropagation Made Simple** — Applying the chain rule to our 3-layer network, updating weights, and watching it learn!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.lines import Line2D

np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

### Activation Function Helper
We will use the **Sigmoid** activation function throughout so every neuron is smooth and differentiable:
$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad \sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$$

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    # Note: if a = sigmoid(z), then derivative is simply a * (1 - a)
    return a * (1 - a)

---
# Part 1: 3-Layer Neural Network (1 Neuron per Layer)

Before jumping into big matrices, let's build the **simplest possible deep network**:
- **1 Input**: $x$
- **Layer 1**: 1 neuron with weight $w_1$, bias $b_1$
- **Layer 2**: 1 neuron with weight $w_2$, bias $b_2$
- **Layer 3 (Output)**: 1 neuron with weight $w_3$, bias $b_3$

Notice in the diagram below that **each bias ($b_1, b_2, b_3$) is drawn as an edge** coming from a constant **`+1` bias node**—because mathematically, adding $b$ is identical to multiplying a constant input `1` by weight $b$ ($1 \cdot b$).

In [ ]:
# Fix random integer weights and biases (between -3 and 3, excluding 0 for clarity)
np.random.seed(7)
possible_ints = [-3, -2, -1, 1, 2, 3]

w1, w2, w3 = np.random.choice(possible_ints, size=3)
b1, b2, b3 = np.random.choice([-1, 0, 1], size=3)

# Single input value
x = 2

print(f"Input x = {x}")
print(f"Layer 1: w1 = {w1}, b1 = {b1}")
print(f"Layer 2: w2 = {w2}, b2 = {b2}")
print(f"Layer 3: w3 = {w3}, b3 = {b3}")

### Visualize Our 1-Neuron-per-Layer Network (with Bias as Edges)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.4))
ax.set_xlim(0, 12.2)
ax.set_ylim(0.5, 4.8)
ax.axis('off')

nodes = [
    (1.2, 1.8, f'Input\nx = {x}', '#dceefb', '#1f77b4'),
    (4.2, 1.8, 'Layer 1\n(σ)', '#fff3cd', '#ff9800'),
    (7.2, 1.8, 'Layer 2\n(σ)', '#ffe0b2', '#ef6c00'),
    (10.2, 1.8, 'Layer 3\n(σ)', '#e8f5e9', '#2e7d32'),
]
weights_labels = [f'w1 = {w1}', f'w2 = {w2}', f'w3 = {w3}']
bias_labels = [f'b1 = {b1}', f'b2 = {b2}', f'b3 = {b3}']
act_labels = ['a1', 'a2', 'y_hat (a3)']

# Draw main nodes
for (nx, ny, text, fc, ec) in nodes:
    ax.add_patch(Circle((nx, ny), 0.62, facecolor=fc, edgecolor=ec, lw=2, zorder=3))
    ax.text(nx, ny, text, ha='center', va='center', fontsize=10, fontweight='bold')

# Draw weight edges between layers
for i in range(3):
    x_start = nodes[i][0] + 0.65
    x_end = nodes[i+1][0] - 0.65
    ax.add_patch(FancyArrowPatch((x_start, 1.8), (x_end, 1.8), arrowstyle='->', mutation_scale=18, lw=2, color='#444', zorder=2))
    ax.text((x_start + x_end)/2, 2.15, weights_labels[i], ha='center', fontsize=11, color='#d62728', fontweight='bold')
    if i > 0:
        ax.text((x_start + x_end)/2, 1.38, f'carries {act_labels[i-1]}', ha='center', fontsize=9, color='#555')

# Draw bias (+1) nodes and bias edges pointing into Layers 1, 2, 3
for i in range(3):
    nx = nodes[i+1][0]
    bx, by = nx - 0.85, 3.85
    ax.add_patch(Circle((bx, by), 0.34, facecolor='#f3e5f5', edgecolor='#7b1fa2', lw=1.8, linestyle='--', zorder=3))
    ax.text(bx, by, '+1', ha='center', va='center', fontsize=10, fontweight='bold', color='#7b1fa2')
    
    ax.add_patch(FancyArrowPatch((bx + 0.16, by - 0.30), (nx - 0.22, 1.8 + 0.58),
                                 arrowstyle='->', mutation_scale=16, lw=1.8, linestyle='--', color='#7b1fa2', zorder=2))
    ax.text(nx - 0.05, 3.15, bias_labels[i], ha='left', fontsize=10.5, color='#7b1fa2', fontweight='bold')

# Final output arrow
ax.annotate('', xy=(11.9, 1.8), xytext=(10.85, 1.8), arrowprops=dict(arrowstyle='->', lw=2, color='#2e7d32'))
ax.text(11.4, 2.15, 'y_hat', ha='center', fontsize=11, fontweight='bold', color='#2e7d32')

plt.title('3-Layer Network (1 Neuron per Layer) — Weights & Bias Shown as Edges', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Step-by-Step Forward Pass
Let's execute **Layer 1**, then **Layer 2**, then **Layer 3** one cell at a time.

In [ ]:
# --- LAYER 1 ---
z1 = w1 * x + b1
a1 = sigmoid(z1)

print(f"z1 = w1 * x + b1 = ({w1}) * ({x}) + ({b1}) = {z1}")
print(f"a1 = sigmoid({z1}) = {a1:.4f}")

In [ ]:
# --- LAYER 2 ---
z2 = w2 * a1 + b2
a2 = sigmoid(z2)

print(f"z2 = w2 * a1 + b2 = ({w2}) * ({a1:.4f}) + ({b2}) = {z2:.4f}")
print(f"a2 = sigmoid({z2:.4f}) = {a2:.4f}")

In [ ]:
# --- LAYER 3 (OUTPUT LAYER) ---
z3 = w3 * a2 + b3
a3 = sigmoid(z3)
y_hat = a3  # Final prediction

print(f"z3 = w3 * a2 + b3 = ({w3}) * ({a2:.4f}) + ({b3}) = {z3:.4f}")
print(f"Final Output y_hat = a3 = sigmoid({z3:.4f}) = {y_hat:.4f}")

### The Complete Forward Pass Equation
If we substitute Layer 1 into Layer 2, and Layer 2 into Layer 3, the **entire 3-layer neural network** is just one nested mathematical function!

$$\hat{y} = \sigma\Big( w_3 \cdot \sigma\big( w_2 \cdot \sigma(w_1 x + b_1) + b_2 \big) + b_3 \Big)$$

Let's verify that plugging our numbers directly into this single equation gives the **exact same output**.

In [ ]:
# Verify with the single nested equation
y_hat_direct = sigmoid(w3 * sigmoid(w2 * sigmoid(w1 * x + b1) + b2) + b3)

print(f"Nested equation with our numbers:")
print(f"y_hat = sigmoid( {w3} * sigmoid( {w2} * sigmoid( {w1}*{x} + {b1} ) + {b2} ) + {b3} )")
print(f"\nStep-by-step output:    {y_hat:.6f}")
print(f"Single-equation output: {y_hat_direct:.6f}")

---
# Part 2: Scaling Up — 3-Layer Network with 2 to 3 Neurons per Layer

Now let's give each layer **multiple neurons**:
- **Input Layer**: 2 features ($x_1, x_2$)
- **Layer 1 (Hidden 1)**: 2 neurons
- **Layer 2 (Hidden 2)**: 3 neurons
- **Layer 3 (Output)**: 2 neurons

Instead of scalar multiplication ($w \cdot x + b$), we use **matrix multiplication** ($A \cdot W + B$). In the diagram, the purple **`+1` bias nodes** connect via dashed edges ($B_1, B_2, B_3$) to every neuron in the next layer.

In [ ]:
# Draw the 2 -> 2 -> 3 -> 2 architecture with Bias (+1) nodes and edges
def draw_network_with_bias(layer_sizes, layer_names, title):
    fig, ax = plt.subplots(figsize=(10.5, 5.6))
    ax.axis('off')
    x_pos = np.linspace(1.2, 8.8, len(layer_sizes))
    colors = ['#dceefb', '#fff3cd', '#ffe0b2', '#e8f5e9']
    edges = ['#1f77b4', '#ff9800', '#ef6c00', '#2e7d32']
    
    coords = []
    for i, n in enumerate(layer_sizes):
        ys = np.linspace(1.3, 4.1, n) if n > 1 else [2.7]
        coords.append([(x_pos[i], y) for y in ys])
        
    # Draw weight edges (solid gray)
    for l in range(len(coords) - 1):
        for (x1, y1) in coords[l]:
            for (x2, y2) in coords[l + 1]:
                ax.plot([x1, x2], [y1, y2], color='#999999', lw=1.2, zorder=1)
        ax.text((x_pos[l] + x_pos[l+1])/2, 1.0, f'W{l+1}', ha='center', fontsize=10, fontweight='bold', color='#555')
                
    # Draw bias (+1) nodes and dashed purple bias edges into the next layer
    bias_y = 5.2
    for l in range(len(coords) - 1):
        bx = x_pos[l]
        ax.add_patch(Circle((bx, bias_y), 0.26, facecolor='#f3e5f5', edgecolor='#7b1fa2', lw=1.8, linestyle='--', zorder=3))
        ax.text(bx, bias_y, '+1', ha='center', va='center', fontsize=9, fontweight='bold', color='#7b1fa2', zorder=4)
        for (x2, y2) in coords[l + 1]:
            ax.plot([bx, x2], [bias_y, y2], color='#7b1fa2', lw=1.2, linestyle='--', alpha=0.75, zorder=1)
        ax.text((bx + x_pos[l+1])/2, 5.05, f'B{l+1}', ha='center', fontsize=10, fontweight='bold', color='#7b1fa2')
                
    # Draw regular neuron nodes
    for l, layer in enumerate(coords):
        for (nx, ny) in layer:
            ax.add_patch(Circle((nx, ny), 0.3, facecolor=colors[l], edgecolor=edges[l], lw=2, zorder=3))
        ax.text(x_pos[l], 0.35, layer_names[l], ha='center', fontsize=10, fontweight='bold', color=edges[l])
        
    legend_elements = [
        Line2D([0], [0], color='#999999', lw=1.5, label='Weight edges (W)'),
        Line2D([0], [0], color='#7b1fa2', lw=1.5, linestyle='--', label='Bias edges (B from +1 node)')
    ]
    ax.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=9.5)
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 5.9)
    plt.title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

draw_network_with_bias(
    [2, 2, 3, 2],
    ['Input\n(2 features)', 'Layer 1\n(2 neurons)', 'Layer 2\n(3 neurons)', 'Layer 3 (Output)\n(2 neurons)'],
    '3-Layer Network (2–3 Neurons per Layer) with Bias Edges'
)

In [ ]:
# Fix small random integer weights and biases for the 2 -> 2 -> 3 -> 2 network
np.random.seed(42)

X_in = np.array([[1, 2]])  # 1 sample, 2 features -> shape (1, 2)

W1 = np.random.randint(-2, 3, size=(2, 2))
B1 = np.random.randint(-1, 2, size=(1, 2))

W2 = np.random.randint(-2, 3, size=(2, 3))
B2 = np.random.randint(-1, 2, size=(1, 3))

W3 = np.random.randint(-2, 3, size=(3, 2))
B3 = np.random.randint(-1, 2, size=(1, 2))

print("Input X_in:", X_in, "shape:", X_in.shape)
print("\nW1 (2x2):\n", W1, "\nB1:", B1)
print("\nW2 (2x3):\n", W2, "\nB2:", B2)
print("\nW3 (3x2):\n", W3, "\nB3:", B3)

### Step-by-Step Matrix Forward Pass
Watch how the shapes flow: `(1, 2) -> (1, 2) -> (1, 3) -> (1, 2)`.

In [ ]:
# --- LAYER 1 (2 neurons) ---
Z1 = X_in @ W1 + B1
A1 = sigmoid(Z1)

print("Z1 = X_in @ W1 + B1 =", Z1, "  shape:", Z1.shape)
print("A1 = sigmoid(Z1)    =", np.round(A1, 4))

In [ ]:
# --- LAYER 2 (3 neurons) ---
Z2 = A1 @ W2 + B2
A2 = sigmoid(Z2)

print("Z2 = A1 @ W2 + B2 =", np.round(Z2, 4), "  shape:", Z2.shape)
print("A2 = sigmoid(Z2)  =", np.round(A2, 4))

In [ ]:
# --- LAYER 3 / OUTPUT (2 neurons) ---
Z3 = A2 @ W3 + B3
A3 = sigmoid(Z3)

print("Z3 = A2 @ W3 + B3      =", np.round(Z3, 4), "  shape:", Z3.shape)
print("Final Output A3 (y_hat)=", np.round(A3, 4))

**Notice the Matrix Equation:**
$$\hat{\mathbf{y}} = \sigma\Big( \sigma\big( \sigma(\mathbf{X} W_1 + B_1) W_2 + B_2 \big) W_3 + B_3 \Big)$$
It is the **exact same structure** as our 1-neuron-per-layer network—just done in parallel for multiple neurons using matrices!

---
# Part 3: How Do We Train It? Introducing the Loss Function

Right now, our weights are **random**, so our network's prediction $\hat{y}$ is just a random guess.

To train the network, we first need a score that measures **how wrong** the prediction is compared to the true target $y$. This is the **Loss Function**.

Let's use **Squared Error Loss** (with a $\frac{1}{2}$ factor to make the derivative clean):
$$L = \frac{1}{2}(\hat{y} - y)^2$$

- If $\hat{y} = y$ (perfect prediction), then $L = 0$.
- The further $\hat{y}$ is from $y$, the larger the loss $L$.
- Its derivative with respect to $\hat{y}$ is simply:
$$\frac{dL}{d\hat{y}} = \hat{y} - y$$

In [ ]:
# Going back to our 1-neuron-per-layer network from Part 1:
# Suppose the true target value we WANT the network to output is y_true = 0.9
y_true = 0.9

loss = 0.5 * (y_hat - y_true) ** 2
print(f"True Target (y):       {y_true:.4f}")
print(f"Current Guess (y_hat): {y_hat:.4f}")
print(f"Error (y_hat - y):     {y_hat - y_true:.4f}")
print(f"Loss L = 0.5*(err)^2:  {loss:.4f}")

In [ ]:
# Visualize the Loss Function as a bowl (just like in Gradient Descent!)
y_hat_range = np.linspace(0, 1.2, 200)
loss_curve = 0.5 * (y_hat_range - y_true) ** 2

plt.figure(figsize=(7, 4))
plt.plot(y_hat_range, loss_curve, lw=2.5, color='#1f77b4', label=r'$L = \frac{1}{2}(\hat{y} - y)^2$')
plt.scatter([y_hat], [loss], color='red', s=100, zorder=5, label=f'Current Prediction ({y_hat:.2f})')
plt.scatter([y_true], [0], color='green', s=100, zorder=5, label=f'Goal / Minimum ({y_true:.2f})')
plt.xlabel('Predicted Output ($\hat{y}$)')
plt.ylabel('Loss (L)')
plt.title('Our Goal: Adjust Weights to Move Down the Loss Bowl')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
# Part 4: The Chain Rule of Derivatives (Intuition First!)

To reduce the Loss $L$, Gradient Descent needs to know:
> *"If I nudge weight $w_1$ slightly, how much does the final Loss $L$ change?"* $\longrightarrow \frac{\partial L}{\partial w_1}$

The challenge: $w_1$ is buried 3 layers deep inside the network! It doesn't touch $L$ directly.

### The Gear Analogy for the Chain Rule
Imagine 3 connected gears: **Gear A $\rightarrow$ Gear B $\rightarrow$ Gear C**.
- If Gear A turns 1 time, Gear B turns **2 times** ($\frac{dB}{dA} = 2$).
- If Gear B turns 1 time, Gear C turns **3 times** ($\frac{dC}{dB} = 3$).
- Therefore, if Gear A turns 1 time, Gear C turns **$2 \times 3 = 6$ times**!

$$\frac{dC}{dA} = \frac{dC}{dB} \cdot \frac{dB}{dA}$$

That's the **entire Chain Rule**: to find how an early variable affects the final output, **multiply the local derivatives along the chain**.

In [ ]:
# Concrete numerical check of the Chain Rule
# Let u = 2 * x, and y = u^2  (so y = (2x)^2 = 4x^2, whose derivative is 8x)
x_val = 3.0

# Forward
u = 2 * x_val
y = u ** 2

# Local derivatives
du_dx = 2          # derivative of (2x) w.r.t. x
dy_du = 2 * u      # derivative of (u^2) w.r.t. u

# Chain Rule: multiply them!
dy_dx_chain = dy_du * du_dx

# Verify by actually nudging x by a tiny epsilon
eps = 1e-6
y_nudged = (2 * (x_val + eps)) ** 2
dy_dx_numerical = (y_nudged - y) / eps

print(f"Chain Rule (dy/du * du/dx): {dy_du} * {du_dx} = {dy_dx_chain:.4f}")
print(f"Numerical nudge check:       {dy_dx_numerical:.4f}")

---
# Part 5: Backpropagation Made Simple

**Backpropagation** is literally just **applying the Chain Rule backward** from the Loss $L$ to each weight ($w_3 \rightarrow w_2 \rightarrow w_1$) and each bias edge ($b_3 \rightarrow b_2 \rightarrow b_1$).

Look at the chain of steps in our 3-layer (1-neuron-per-layer) network:
$$x \xrightarrow{w_1, b_1} z_1 \xrightarrow{\sigma} a_1 \xrightarrow{w_2, b_2} z_2 \xrightarrow{\sigma} a_2 \xrightarrow{w_3, b_3} z_3 \xrightarrow{\sigma} a_3 (\hat{y}) \longrightarrow L$$

Now walk **right to left** (from $L$ back to $w_1$), multiplying local derivatives as we go:

1. **Layer 3 (Output):**
$$\delta_3 = \frac{\partial L}{\partial z_3} = \underbrace{\frac{\partial L}{\partial a_3}}_{(a_3 - y)} \cdot \underbrace{\frac{\partial a_3}{\partial z_3}}_{a_3(1 - a_3)} \qquad \Longrightarrow \qquad \frac{\partial L}{\partial w_3} = \delta_3 \cdot a_2, \quad \frac{\partial L}{\partial b_3} = \delta_3 \cdot 1$$

2. **Layer 2 (Pass $\delta_3$ backward through $w_3$):**
$$\delta_2 = \frac{\partial L}{\partial z_2} = (\delta_3 \cdot w_3) \cdot a_2(1 - a_2) \qquad \Longrightarrow \qquad \frac{\partial L}{\partial w_2} = \delta_2 \cdot a_1, \quad \frac{\partial L}{\partial b_2} = \delta_2 \cdot 1$$

3. **Layer 1 (Pass $\delta_2$ backward through $w_2$):**
$$\delta_1 = \frac{\partial L}{\partial z_1} = (\delta_2 \cdot w_2) \cdot a_1(1 - a_1) \qquad \Longrightarrow \qquad \frac{\partial L}{\partial w_1} = \delta_1 \cdot x, \quad \frac{\partial L}{\partial b_1} = \delta_1 \cdot 1$$

Notice why $\frac{\partial L}{\partial b_k} = \delta_k \cdot 1$: because the bias edge comes from a constant **`+1` node**!

In [ ]:
# Visual diagram: Forward Pass (blue), Bias Edges (purple), and Backward Pass (red)
fig, ax = plt.subplots(figsize=(11.5, 4.8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 5.2)
ax.axis('off')

positions = [1.0, 3.4, 5.8, 8.2, 10.6]
labels = ['Input\nx', 'Layer 1\n(σ)', 'Layer 2\n(σ)', 'Layer 3\n(σ)', 'Loss\nL']
fcs = ['#dceefb', '#fff3cd', '#ffe0b2', '#e8f5e9', '#ffebee']
ecs = ['#1f77b4', '#ff9800', '#ef6c00', '#2e7d32', '#c62828']

for px, lbl, fc, ec in zip(positions, labels, fcs, ecs):
    ax.add_patch(Circle((px, 2.0), 0.56, facecolor=fc, edgecolor=ec, lw=2, zorder=3))
    ax.text(px, 2.0, lbl, ha='center', va='center', fontsize=9.5, fontweight='bold')

# Forward weight edges (top arcs)
fwd_labels = ['w1 (x)', 'w2 (a1)', 'w3 (a2)', 'y_hat']
for i in range(4):
    ax.annotate('', xy=(positions[i+1]-0.52, 2.38), xytext=(positions[i]+0.52, 2.38),
                arrowprops=dict(arrowstyle='->', lw=2, color='#1f77b4', connectionstyle='arc3,rad=-0.22'))
    ax.text((positions[i]+positions[i+1])/2, 2.95, fwd_labels[i], ha='center', fontsize=9.5, color='#1f77b4', fontweight='bold')

# Bias (+1) nodes and bias edges entering Layers 1, 2, 3 from above
for idx, layer_i in enumerate([1, 2, 3], start=1):
    px = positions[layer_i]
    ax.add_patch(Circle((px, 4.25), 0.30, facecolor='#f3e5f5', edgecolor='#7b1fa2', lw=1.8, linestyle='--', zorder=3))
    ax.text(px, 4.25, '+1', ha='center', va='center', fontsize=9, fontweight='bold', color='#7b1fa2')
    ax.annotate('', xy=(px, 2.58), xytext=(px, 3.93),
                arrowprops=dict(arrowstyle='->', lw=1.8, linestyle='--', color='#7b1fa2'))
    ax.text(px + 0.12, 3.42, f'b{idx}', ha='left', fontsize=9.5, color='#7b1fa2', fontweight='bold')
    ax.text(px - 0.12, 3.42, f'dL/db{idx}', ha='right', fontsize=8.5, color='#c62828', fontweight='bold')

# Backward gradient flow (bottom arcs)
bwd_labels = ['dL/dw1', 'dL/dw2', 'dL/dw3', 'dL/dy_hat']
for i in range(4):
    ax.annotate('', xy=(positions[i]+0.52, 1.52), xytext=(positions[i+1]-0.52, 1.52),
                arrowprops=dict(arrowstyle='->', lw=2, color='#c62828', connectionstyle='arc3,rad=-0.22'))
    ax.text((positions[i]+positions[i+1])/2, 0.78, bwd_labels[i], ha='center', fontsize=9.5, color='#c62828', fontweight='bold')

ax.text(5.8, 4.9, 'FORWARD PASS (Weights + Bias Edges) ->', ha='center', fontsize=11, fontweight='bold', color='#1f77b4')
ax.text(5.8, 0.22, '<- BACKWARD PASS (Chain Rule): Gradients Flow to Every Weight & Bias Edge', ha='center', fontsize=10.5, fontweight='bold', color='#c62828')

plt.tight_layout()
plt.show()

### Step-by-Step Backpropagation in Code (1-Neuron-per-Layer Network)
Let's compute the exact gradients for Layer 3, Layer 2, and Layer 1 one cell at a time.

In [ ]:
# --- BACKWARD STEP 1: LAYER 3 ---
dL_da3 = a3 - y_true
da3_dz3 = sigmoid_derivative(a3)
delta3 = dL_da3 * da3_dz3

dL_dw3 = delta3 * a2
dL_db3 = delta3 * 1  # bias edge comes from +1

print(f"delta3 = {delta3:.6f}")
print(f"dL/dw3 = {dL_dw3:.6f},  dL/db3 = {dL_db3:.6f}")

In [ ]:
# --- BACKWARD STEP 2: LAYER 2 ---
delta2 = (delta3 * w3) * sigmoid_derivative(a2)

dL_dw2 = delta2 * a1
dL_db2 = delta2 * 1

print(f"delta2 = {delta2:.6f}")
print(f"dL/dw2 = {dL_dw2:.6f},  dL/db2 = {dL_db2:.6f}")

In [ ]:
# --- BACKWARD STEP 3: LAYER 1 ---
delta1 = (delta2 * w2) * sigmoid_derivative(a1)

dL_dw1 = delta1 * x
dL_db1 = delta1 * 1

print(f"delta1 = {delta1:.6f}")
print(f"dL/dw1 = {dL_dw1:.6f},  dL/db1 = {dL_db1:.6f}")

### One Gradient Descent Update Step
Now we update every weight and bias edge in the **opposite direction of its gradient**:
$$w_{\text{new}} = w_{\text{old}} - \eta \cdot \frac{\partial L}{\partial w}$$
Let's do **just 1 update** and verify that the Loss goes **down**!

In [ ]:
lr = 0.5  # learning rate (eta)

# Update weights and biases
w1_new, b1_new = w1 - lr * dL_dw1, b1 - lr * dL_db1
w2_new, b2_new = w2 - lr * dL_dw2, b2 - lr * dL_db2
w3_new, b3_new = w3 - lr * dL_dw3, b3 - lr * dL_db3

# Run Forward Pass again with updated weights
a1_new = sigmoid(w1_new * x + b1_new)
a2_new = sigmoid(w2_new * a1_new + b2_new)
y_hat_new = sigmoid(w3_new * a2_new + b3_new)
loss_new = 0.5 * (y_hat_new - y_true) ** 2

print(f"Before 1 step -> Prediction: {y_hat:.4f} | Loss: {loss:.6f}")
print(f"After  1 step -> Prediction: {y_hat_new:.4f} | Loss: {loss_new:.6f}")
print(f"Target y_true -> {y_true:.4f}")

### Putting It All Together: Full Training Loop (100 Epochs)
Now let's repeat `Forward Pass -> Compute Loss -> Backpropagation -> Weight Update` for 100 steps and watch the network learn to hit `y_true = 0.9`!

In [ ]:
# Start fresh from our original integer weights
tw1, tb1 = float(w1), float(b1)
tw2, tb2 = float(w2), float(b2)
tw3, tb3 = float(w3), float(b3)

lr = 1.0
epochs = 100
loss_history = []
pred_history = []

for epoch in range(epochs):
    # 1. Forward pass
    ta1 = sigmoid(tw1 * x + tb1)
    ta2 = sigmoid(tw2 * ta1 + tb2)
    ta3 = sigmoid(tw3 * ta2 + tb3)
    
    # 2. Loss
    tloss = 0.5 * (ta3 - y_true) ** 2
    loss_history.append(tloss)
    pred_history.append(ta3)
    
    # 3. Backpropagation (Chain Rule)
    d3 = (ta3 - y_true) * sigmoid_derivative(ta3)
    d2 = (d3 * tw3) * sigmoid_derivative(ta2)
    d1 = (d2 * tw2) * sigmoid_derivative(ta1)
    
    # 4. Gradient Descent Update
    tw3 -= lr * (d3 * ta2); tb3 -= lr * d3
    tw2 -= lr * (d2 * ta1); tb2 -= lr * d2
    tw1 -= lr * (d1 * x);   tb1 -= lr * d1

print(f"Epoch   0 -> Prediction: {pred_history[0]:.4f} | Loss: {loss_history[0]:.6f}")
print(f"Epoch 100 -> Prediction: {pred_history[-1]:.4f} | Loss: {loss_history[-1]:.6f}")

In [ ]:
# Plot Loss and Prediction Convergence side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(loss_history, color='#d62728', lw=2.5)
axes[0].set_title('Loss Decreasing Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Squared Error Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(pred_history, color='#1f77b4', lw=2.5, label='Network Prediction ($\hat{y}$)')
axes[1].axhline(y_true, color='green', linestyle='--', lw=2, label=f'Target ($y = {y_true}$)')
axes[1].set_title('Prediction Converging to Target')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Output Value')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()